# 04 — Prompt Prototyping

This notebook builds and tests the extraction prompt templates before extracting them to
`ave/prompts/templates.py`. Everything runs **locally** — no GPU or API calls needed.

**What we do here:**

1. Load the split data and schemas from notebooks 02–03
2. Build the zero-shot prompt template (Variant A) using Jinja2
3. Render it for sample records across all 5 categories — inspect the output
4. Estimate token counts with `tiktoken` to understand prompt budget
5. Build the few-shot example selector for Variant C (2–5 examples from train split)
6. Render few-shot prompts and compare token overhead vs zero-shot
7. Build the fine-tuning instruction format (Variant D) — system/user/assistant turns
8. Summarize decisions: final prompt wording, few-shot selection strategy, token budgets

In [1]:
import json
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
import tiktoken
from jinja2 import Template

CLEANED_DIR = Path("../data/WDC-PAVE/cleaned")

## 1. Load data

In [2]:
def load_jsonl(path: Path) -> list[dict]:
    with open(path) as f:
        return [json.loads(line) for line in f]

train = load_jsonl(CLEANED_DIR / "train.jsonl")
val = load_jsonl(CLEANED_DIR / "val.jsonl")
test = load_jsonl(CLEANED_DIR / "test.jsonl")

with open(CLEANED_DIR / "category_schemas.json") as f:
    CATEGORY_SCHEMAS = json.load(f)

print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")
print(f"Categories: {list(CATEGORY_SCHEMAS.keys())}")

Train: 994 | Val: 142 | Test: 284
Categories: ['Computers And Accessories', 'Home And Garden', 'Office Products', 'Jewelry', 'Grocery And Gourmet Food']


## 2. Zero-shot prompt template (Variant A)

This is the core prompt from the plan (`AVE-plan/03-prompt-and-variants.md`).
We use Jinja2 so the same template works for rendering and for `ave/prompts/templates.py`.

In [3]:
ZERO_SHOT_TEMPLATE_STR = """\
You are an information extraction system.

Task:
Extract the product attributes from the product title and description.

Category:
{{ category }}

Allowed attributes:
{{ attribute_list }}

Rules:
- Return exactly one valid JSON object.
- Use exactly the attribute names above as keys.
- If an attribute is missing or cannot be determined from the text, return null.
- If multiple values apply to an attribute, return them as a JSON array.
- Do not add extra keys.
- Do not explain anything.
- Use only information from the input text.
- Do not guess.
- The word "Null" appearing in the input text is a data artifact, not a product attribute value. Ignore it.

Input title:
{{ title }}

Input description:
{{ description }}"""

ZERO_SHOT_TEMPLATE = Template(ZERO_SHOT_TEMPLATE_STR)


def render_zero_shot(record: dict, schemas: dict[str, list[str]]) -> str:
    """Render the zero-shot extraction prompt for a single record."""
    category = record["category"]
    attrs = schemas[category]
    return ZERO_SHOT_TEMPLATE.render(
        category=category,
        attribute_list="\n".join(f"- {a}" for a in attrs),
        title=record["input_title"],
        description=record["input_description"],
    )

print("Template defined. Let's render it for a sample record.")

Template defined. Let's render it for a sample record.


### 2.1 Render zero-shot prompt for one record per category

Let's see exactly what the model would receive. Check that the category, attribute list,
title, and description all look correct.

In [4]:
# Pick one train record per category and render the full prompt
for cat in CATEGORY_SCHEMAS:
    rec = next(r for r in train if r["category"] == cat)
    prompt = render_zero_shot(rec, CATEGORY_SCHEMAS)
    print(f"{'='*80}")
    print(f"CATEGORY: {cat} | ID: {rec['id']}")
    print(f"{'='*80}")
    print(prompt)
    print(f"\n--- Expected gold JSON ---")
    print(json.dumps(rec["gold_json"], indent=2))
    print(f"\n{'='*80}\n")

CATEGORY: Computers And Accessories | ID: 13803205
You are an information extraction system.

Task:
Extract the product attributes from the product title and description.

Category:
Computers And Accessories

Allowed attributes:
- Generation
- Part Number
- Product Type
- Cache
- Processor Type
- Processor Core
- Interface
- Manufacturer
- Capacity
- Ports
- Rotational Speed

Rules:
- Return exactly one valid JSON object.
- Use exactly the attribute names above as keys.
- If an attribute is missing or cannot be determined from the text, return null.
- If multiple values apply to an attribute, return them as a JSON array.
- Do not add extra keys.
- Do not explain anything.
- Use only information from the input text.
- Do not guess.
- The word "Null" appearing in the input text is a data artifact, not a product attribute value. Ignore it.

Input title:
653957-001 HP G8 G9 600-GB 6G 10K 2.5 SAS SC

Input description:
Description:HP 600GB 2.5-inch SFF Serial Attached SCSI (SAS)6G 10K Enter

## 3. Token count estimation

We use `tiktoken` (cl100k_base — GPT-4/Claude-class tokenizer) to estimate prompt sizes.
This helps us understand:
- How much context budget the prompt uses
- Whether few-shot examples will fit within typical context windows
- What the token cost will look like for API-based variants (E)

In [5]:
enc = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    return len(enc.encode(text))

# Count tokens for every record in every split (zero-shot prompt)
token_rows = []
for split_name, split_data in [("train", train), ("val", val), ("test", test)]:
    for rec in split_data:
        prompt = render_zero_shot(rec, CATEGORY_SCHEMAS)
        gold_str = json.dumps(rec["gold_json"])
        token_rows.append({
            "split": split_name,
            "category": rec["category"],
            "prompt_tokens": count_tokens(prompt),
            "gold_tokens": count_tokens(gold_str),
            "total_tokens": count_tokens(prompt) + count_tokens(gold_str),
        })

tok_df = pd.DataFrame(token_rows)

print("Zero-shot prompt token statistics (all splits):\n")
print(tok_df[["prompt_tokens", "gold_tokens", "total_tokens"]].describe().round(0))
print(f"\nMax prompt tokens: {tok_df['prompt_tokens'].max()}")
print(f"Max gold tokens:   {tok_df['gold_tokens'].max()}")
print(f"Max total tokens:  {tok_df['total_tokens'].max()}")

Zero-shot prompt token statistics (all splits):

       prompt_tokens  gold_tokens  total_tokens
count         1420.0       1420.0        1420.0
mean           275.0         66.0         341.0
std             44.0         22.0          51.0
min            183.0         23.0         206.0
25%            242.0         51.0         308.0
50%            269.0         72.0         337.0
75%            304.0         82.0         372.0
max            423.0        154.0         549.0

Max prompt tokens: 423
Max gold tokens:   154
Max total tokens:  549


In [6]:
# Token counts per category — categories with more attributes or longer descriptions cost more
print("Mean token counts by category:\n")
print(tok_df.groupby("category")[["prompt_tokens", "gold_tokens", "total_tokens"]]
      .mean().round(0).sort_values("prompt_tokens", ascending=False))

Mean token counts by category:

                           prompt_tokens  gold_tokens  total_tokens
category                                                           
Office Products                    290.0         77.0         367.0
Jewelry                            285.0         29.0         315.0
Computers And Accessories          271.0         87.0         358.0
Grocery And Gourmet Food           263.0         41.0         305.0
Home And Garden                    263.0         64.0         327.0


In [7]:
# Break down: how many tokens come from each part of the prompt?
# This helps us understand where the budget is going.
rec = next(r for r in train if r["category"] == "Computers And Accessories")

# Measure each section separately
parts = {
    "system_preamble": "You are an information extraction system.\n\nTask:\nExtract the product attributes from the product title and description.",
    "category": f"Category:\n{rec['category']}",
    "attribute_list": "Allowed attributes:\n" + "\n".join(f"- {a}" for a in CATEGORY_SCHEMAS[rec["category"]]),
    "rules": """Rules:
- Return exactly one valid JSON object.
- Use exactly the attribute names above as keys.
- If an attribute is missing or cannot be determined from the text, return null.
- If multiple values apply to an attribute, return them as a JSON array.
- Do not add extra keys.
- Do not explain anything.
- Use only information from the input text.
- Do not guess.
- The word "Null" appearing in the input text is a data artifact, not a product attribute value. Ignore it.""",
    "title": f"Input title:\n{rec['input_title']}",
    "description": f"Input description:\n{rec['input_description']}",
}

print(f"Token breakdown for a Computers And Accessories record (ID={rec['id']}):\n")
total = 0
for part_name, part_text in parts.items():
    n = count_tokens(part_text)
    total += n
    print(f"  {part_name:25s} {n:>4} tokens")
print(f"  {'---':25s} {'----':>4}")
print(f"  {'TOTAL':25s} {total:>4} tokens")

Token breakdown for a Computers And Accessories record (ID=13803205):

  system_preamble             20 tokens
  category                     6 tokens
  attribute_list              41 tokens
  rules                      105 tokens
  title                       28 tokens
  description                139 tokens
  ---                       ----
  TOTAL                      339 tokens


## 4. Few-shot example selector (Variant C)

The plan specifies 2–5 in-context examples from the **train split only** (never val or test).

**Selection strategy:** same-category examples, since the attribute schema differs by category.
We pick examples that have a good mix of filled and null attributes to show the model both
extraction and null-handling patterns.

In [8]:
# Index train records by category for fast lookup
train_by_cat: dict[str, list[dict]] = defaultdict(list)
for rec in train:
    train_by_cat[rec["category"]].append(rec)

print("Train records per category:")
for cat, recs in sorted(train_by_cat.items()):
    print(f"  {cat}: {len(recs)}")

Train records per category:
  Computers And Accessories: 305
  Grocery And Gourmet Food: 57
  Home And Garden: 250
  Jewelry: 175
  Office Products: 207


In [9]:
def compute_fill_ratio(record: dict) -> float:
    """Fraction of non-null attributes in the gold JSON."""
    gold = record["gold_json"]
    filled = sum(1 for v in gold.values() if v is not None)
    return filled / len(gold)


def select_few_shot_examples(
    category: str,
    train_by_cat: dict[str, list[dict]],
    n: int = 3,
    exclude_id: int | None = None,
) -> list[dict]:
    """Select n diverse few-shot examples from the train split for a given category.
    
    Strategy:
    - Sort by fill ratio
    - Pick from evenly spaced quantiles to get a mix of sparse and dense examples
    - Exclude the target record if it happens to be in the pool (shouldn't happen
      since we only use train examples for val/test, but just in case)
    """
    pool = [r for r in train_by_cat[category] if r["id"] != exclude_id]
    # Sort by fill ratio
    pool.sort(key=compute_fill_ratio)
    
    if len(pool) <= n:
        return pool
    
    # Pick evenly spaced indices
    step = len(pool) / n
    indices = [int(i * step + step / 2) for i in range(n)]
    return [pool[i] for i in indices]


# Test: select 3 examples for each category, show their fill ratios
for cat in CATEGORY_SCHEMAS:
    examples = select_few_shot_examples(cat, train_by_cat, n=3)
    ratios = [compute_fill_ratio(ex) for ex in examples]
    print(f"{cat}:")
    for ex, r in zip(examples, ratios):
        n_filled = sum(1 for v in ex["gold_json"].values() if v is not None)
        n_total = len(ex["gold_json"])
        print(f"  ID={ex['id']}, fill={n_filled}/{n_total} ({r:.0%})")
    print()

Computers And Accessories:
  ID=1236633, fill=4/11 (36%)
  ID=9254829, fill=6/11 (55%)
  ID=10487333, fill=7/11 (64%)

Home And Garden:
  ID=10093114, fill=2/8 (25%)
  ID=6962837, fill=4/8 (50%)
  ID=4511761, fill=5/8 (62%)

Office Products:
  ID=15556244, fill=4/10 (40%)
  ID=7924863, fill=6/10 (60%)
  ID=3425944, fill=7/10 (70%)

Jewelry:
  ID=16422221, fill=3/3 (100%)
  ID=3404407, fill=3/3 (100%)
  ID=193174, fill=3/3 (100%)

Grocery And Gourmet Food:
  ID=14034427, fill=3/5 (60%)
  ID=7764333, fill=4/5 (80%)
  ID=13033950, fill=5/5 (100%)



### 4.1 Build the few-shot prompt template

The few-shot prompt wraps the zero-shot template with examples before the target input.
Each example shows: title → description → gold JSON.

In [10]:
FEW_SHOT_TEMPLATE_STR = """\
You are an information extraction system.

Task:
Extract the product attributes from the product title and description.

Category:
{{ category }}

Allowed attributes:
{{ attribute_list }}

Rules:
- Return exactly one valid JSON object.
- Use exactly the attribute names above as keys.
- If an attribute is missing or cannot be determined from the text, return null.
- If multiple values apply to an attribute, return them as a JSON array.
- Do not add extra keys.
- Do not explain anything.
- Use only information from the input text.
- Do not guess.
- The word "Null" appearing in the input text is a data artifact, not a product attribute value. Ignore it.

{% for ex in examples %}
Example {{ loop.index }}:

Input title:
{{ ex.input_title }}

Input description:
{{ ex.input_description }}

Output:
{{ ex.gold_json_str }}

{% endfor %}
Now extract attributes from this product:

Input title:
{{ title }}

Input description:
{{ description }}"""

FEW_SHOT_TEMPLATE = Template(FEW_SHOT_TEMPLATE_STR)


def render_few_shot(
    record: dict,
    schemas: dict[str, list[str]],
    train_by_cat: dict[str, list[dict]],
    n_examples: int = 3,
) -> str:
    """Render the few-shot extraction prompt for a single record."""
    category = record["category"]
    attrs = schemas[category]
    examples = select_few_shot_examples(category, train_by_cat, n=n_examples, exclude_id=record["id"])
    
    # Prepare examples with formatted gold JSON
    ex_data = []
    for ex in examples:
        ex_data.append({
            "input_title": ex["input_title"],
            "input_description": ex["input_description"],
            "gold_json_str": json.dumps(ex["gold_json"], indent=2),
        })
    
    return FEW_SHOT_TEMPLATE.render(
        category=category,
        attribute_list="\n".join(f"- {a}" for a in attrs),
        examples=ex_data,
        title=record["input_title"],
        description=record["input_description"],
    )

print("Few-shot template defined.")

Few-shot template defined.


### 4.2 Render and inspect a few-shot prompt

Let's see the full prompt for one record — including the examples — to check it reads well.

In [11]:
# Render a few-shot prompt for a val record (simulates real usage — val/test records get
# few-shot examples from the train split)
val_rec = next(r for r in val if r["category"] == "Office Products")
fs_prompt = render_few_shot(val_rec, CATEGORY_SCHEMAS, train_by_cat, n_examples=3)

print(f"Category: {val_rec['category']} | ID: {val_rec['id']}")
print(f"Prompt length: {len(fs_prompt)} chars, {count_tokens(fs_prompt)} tokens")
print(f"\n{'='*80}")
print(fs_prompt)
print(f"{'='*80}")
print(f"\n--- Expected gold JSON ---")
print(json.dumps(val_rec["gold_json"], indent=2))

Category: Office Products | ID: 6361626
Prompt length: 3781 chars, 991 tokens

You are an information extraction system.

Task:
Extract the product attributes from the product title and description.

Category:
Office Products

Allowed attributes:
- Product Type
- Color(s)
- Pack Quantity
- Length
- Width
- Height
- Depth
- Paper Weight
- Manufacturer Stock Number
- Retail UPC

Rules:
- Return exactly one valid JSON object.
- Use exactly the attribute names above as keys.
- If an attribute is missing or cannot be determined from the text, return null.
- If multiple values apply to an attribute, return them as a JSON array.
- Do not add extra keys.
- Do not explain anything.
- Use only information from the input text.
- Do not guess.
- The word "Null" appearing in the input text is a data artifact, not a product attribute value. Ignore it.


Example 1:

Input title:
Acco Quartet MP1650Q Class 3 Metal Laser Pointer w/Pocket Clip Acco Clip - QRTMP1650Q ReStockIt

Input description:
Class 3

### 4.3 Few-shot token overhead

How much do the in-context examples add? Compare zero-shot vs few-shot (2, 3, 5 examples).

In [12]:
# For each category, measure token counts at different n_examples
overhead_rows = []
for cat in CATEGORY_SCHEMAS:
    # Pick a representative val record
    rec = next(r for r in val if r["category"] == cat)
    zs_tokens = count_tokens(render_zero_shot(rec, CATEGORY_SCHEMAS))
    
    for n_ex in [2, 3, 5]:
        fs_tokens = count_tokens(render_few_shot(rec, CATEGORY_SCHEMAS, train_by_cat, n_examples=n_ex))
        overhead_rows.append({
            "category": cat,
            "n_examples": n_ex,
            "zero_shot_tokens": zs_tokens,
            "few_shot_tokens": fs_tokens,
            "overhead_tokens": fs_tokens - zs_tokens,
            "overhead_pct": round((fs_tokens - zs_tokens) / zs_tokens * 100, 1),
        })

oh_df = pd.DataFrame(overhead_rows)
print("Few-shot token overhead by category and number of examples:\n")
print(oh_df.pivot_table(
    index="category",
    columns="n_examples",
    values=["few_shot_tokens", "overhead_pct"],
).to_string())

Few-shot token overhead by category and number of examples:

                          few_shot_tokens                overhead_pct              
n_examples                              2      3       5            2      3      5
category                                                                           
Computers And Accessories           769.0  951.0  1463.0        100.3  147.7  281.0
Grocery And Gourmet Food            760.0  810.0   948.0        172.4  190.3  239.8
Home And Garden                     580.0  768.0  1244.0        144.7  224.1  424.9
Jewelry                             727.0  777.0  1185.0        134.5  150.6  282.3
Office Products                     874.0  991.0  1308.0        178.3  215.6  316.6


In [13]:
# Summary: will the few-shot prompts fit in common context windows?
max_fs5 = oh_df[oh_df["n_examples"] == 5]["few_shot_tokens"].max()
print(f"Max few-shot (5 examples) prompt tokens: {max_fs5}")
print()
for ctx in [2048, 4096, 8192]:
    headroom = ctx - max_fs5
    fits = "YES" if headroom > 200 else "NO — too tight"
    print(f"  {ctx:,}-token context: {headroom:,} tokens headroom for output → {fits}")

Max few-shot (5 examples) prompt tokens: 1463

  2,048-token context: 585 tokens headroom for output → YES
  4,096-token context: 2,633 tokens headroom for output → YES
  8,192-token context: 6,729 tokens headroom for output → YES


## 5. Constrained decoding schema (Variant B)

Variant B uses the same zero-shot prompt but constrains the output to valid JSON matching
the category schema. We build the JSON Schema here so it can be passed to `outlines` or
an API's structured output parameter.

In [14]:
def build_json_schema(category: str, schemas: dict[str, list[str]]) -> dict:
    """Build a JSON Schema for constrained decoding of a category's gold JSON.
    
    Each attribute value can be:
    - null (missing)
    - a string (single value)
    - an array of strings (multi-value)
    """
    attrs = schemas[category]
    properties = {}
    for attr in attrs:
        properties[attr] = {
            "oneOf": [
                {"type": "null"},
                {"type": "string"},
                {"type": "array", "items": {"type": "string"}, "minItems": 2},
            ]
        }
    
    return {
        "type": "object",
        "properties": properties,
        "required": attrs,
        "additionalProperties": False,
    }


# Show the schema for each category
for cat in CATEGORY_SCHEMAS:
    schema = build_json_schema(cat, CATEGORY_SCHEMAS)
    print(f"=== {cat} ({len(schema['required'])} required keys) ===")
    print(json.dumps(schema, indent=2)[:500])
    print("...\n")

=== Computers And Accessories (11 required keys) ===
{
  "type": "object",
  "properties": {
    "Generation": {
      "oneOf": [
        {
          "type": "null"
        },
        {
          "type": "string"
        },
        {
          "type": "array",
          "items": {
            "type": "string"
          },
          "minItems": 2
        }
      ]
    },
    "Part Number": {
      "oneOf": [
        {
          "type": "null"
        },
        {
          "type": "string"
        },
        {
          "type": "array",
          "
...

=== Home And Garden (8 required keys) ===
{
  "type": "object",
  "properties": {
    "Product Type": {
      "oneOf": [
        {
          "type": "null"
        },
        {
          "type": "string"
        },
        {
          "type": "array",
          "items": {
            "type": "string"
          },
          "minItems": 2
        }
      ]
    },
    "Color": {
      "oneOf": [
        {
          "type": "null"
        },

In [15]:
# Validate that our gold JSONs actually pass the JSON Schema
# This is a critical check — if gold doesn't match the schema, constrained decoding
# would force the model into an impossible situation
import jsonschema

errors = []
all_data = train + val + test
for rec in all_data:
    schema = build_json_schema(rec["category"], CATEGORY_SCHEMAS)
    try:
        jsonschema.validate(rec["gold_json"], schema)
    except jsonschema.ValidationError as e:
        errors.append((rec["id"], rec["category"], str(e.message)[:100]))

if errors:
    print(f"VALIDATION FAILED: {len(errors)} gold JSONs don't match the JSON Schema!")
    for rec_id, cat, msg in errors[:10]:
        print(f"  ID={rec_id} ({cat}): {msg}")
else:
    print(f"All {len(all_data):,} gold JSONs pass JSON Schema validation.")

All 1,420 gold JSONs pass JSON Schema validation.


## 6. Fine-tuning instruction format (Variant D)

For supervised fine-tuning, we format each example as a chat-style conversation:
- **System message:** task description + rules (shared across examples)
- **User message:** category, attributes, title, description
- **Assistant message:** the gold JSON

This format works with most SFT frameworks (TRL, Axolotl, etc.).

In [16]:
SYSTEM_MESSAGE = """\
You are an information extraction system. Extract product attributes from the given title and description.

Rules:
- Return exactly one valid JSON object.
- Use exactly the attribute names listed by the user as keys.
- If an attribute is missing or cannot be determined, return null.
- If multiple values apply, return them as a JSON array sorted alphabetically.
- Do not add extra keys. Do not explain. Use only information from the input text.
- The word "Null" in the input is a data artifact — ignore it."""


USER_TEMPLATE_STR = """\
Category: {{ category }}

Attributes:
{{ attribute_list }}

Title: {{ title }}

Description: {{ description }}"""

USER_TEMPLATE = Template(USER_TEMPLATE_STR)


def build_sft_example(record: dict, schemas: dict[str, list[str]]) -> dict:
    """Build a chat-format SFT training example."""
    category = record["category"]
    attrs = schemas[category]
    
    user_msg = USER_TEMPLATE.render(
        category=category,
        attribute_list=", ".join(attrs),
        title=record["input_title"],
        description=record["input_description"],
    )
    
    assistant_msg = json.dumps(record["gold_json"])
    
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_MESSAGE},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": assistant_msg},
        ]
    }


# Show one example
sft_example = build_sft_example(train[0], CATEGORY_SCHEMAS)
for msg in sft_example["messages"]:
    print(f"--- {msg['role'].upper()} ---")
    print(msg["content"][:300])
    if len(msg["content"]) > 300:
        print("...")
    print()

--- SYSTEM ---
You are an information extraction system. Extract product attributes from the given title and description.

Rules:
- Return exactly one valid JSON object.
- Use exactly the attribute names listed by the user as keys.
- If an attribute is missing or cannot be determined, return null.
- If multiple va
...

--- USER ---
Category: Home And Garden

Attributes:
Product Type, Color, Length, Width, Height, Depth, Manufacturer Stock Number, Retail UPC

Title: Pneumatic Lift Lab Stools w/Back, Height-Adjustable, 17-22, Black w/Back by Safco SAF3430BL - OnTimeSupplies.com

Description: Pneumatic lift adjusts to accommodate
...

--- ASSISTANT ---
{"Product Type": "Furniture, Storage, Racks and Fixtures", "Color": "Black", "Length": null, "Width": null, "Height": "99.1", "Depth": null, "Manufacturer Stock Number": "SAF3430BL", "Retail UPC": null}



In [17]:
# Token count for SFT examples — this determines fine-tuning cost and max_seq_len
sft_token_rows = []
for rec in train:
    example = build_sft_example(rec, CATEGORY_SCHEMAS)
    total_tokens = sum(count_tokens(msg["content"]) for msg in example["messages"])
    system_tokens = count_tokens(example["messages"][0]["content"])
    user_tokens = count_tokens(example["messages"][1]["content"])
    assistant_tokens = count_tokens(example["messages"][2]["content"])
    sft_token_rows.append({
        "category": rec["category"],
        "system_tokens": system_tokens,
        "user_tokens": user_tokens,
        "assistant_tokens": assistant_tokens,
        "total_tokens": total_tokens,
    })

sft_tok_df = pd.DataFrame(sft_token_rows)
print("SFT example token statistics (train split):\n")
print(sft_tok_df[["system_tokens", "user_tokens", "assistant_tokens", "total_tokens"]].describe().round(0))
print(f"\nMax total tokens: {sft_tok_df['total_tokens'].max()}")
print(f"Sum total tokens (full train set): {sft_tok_df['total_tokens'].sum():,}")

SFT example token statistics (train split):

       system_tokens  user_tokens  assistant_tokens  total_tokens
count          994.0        994.0             994.0         994.0
mean           106.0        138.0              66.0         310.0
std              0.0         44.0              22.0          50.0
min            106.0         50.0              23.0         179.0
25%            106.0        104.0              50.0         277.0
50%            106.0        132.0              72.0         307.0
75%            106.0        168.0              82.0         341.0
max            106.0        286.0             154.0         516.0

Max total tokens: 516
Sum total tokens (full train set): 308,284


In [18]:
# Per-category breakdown
print("Mean SFT token counts by category:\n")
print(sft_tok_df.groupby("category")[["user_tokens", "assistant_tokens", "total_tokens"]]
      .mean().round(0).sort_values("total_tokens", ascending=False))

Mean SFT token counts by category:

                           user_tokens  assistant_tokens  total_tokens
category                                                              
Office Products                  151.0              77.0         334.0
Computers And Accessories        132.0              87.0         326.0
Home And Garden                  125.0              64.0         295.0
Jewelry                          151.0              29.0         286.0
Grocery And Gourmet Food         133.0              41.0         280.0


## 7. Compare all prompt variants side by side

Let's pick one record and show what each variant produces as input to the model.

In [19]:
# Pick a val record for comparison
sample_rec = next(r for r in val if r["category"] == "Computers And Accessories")

# Zero-shot (Variant A + B use the same prompt, B adds constrained decoding)
zs = render_zero_shot(sample_rec, CATEGORY_SCHEMAS)
# Few-shot (Variant C)
fs3 = render_few_shot(sample_rec, CATEGORY_SCHEMAS, train_by_cat, n_examples=3)
# SFT format (Variant D — this is the training format, at inference we use system + user only)
sft = build_sft_example(sample_rec, CATEGORY_SCHEMAS)
sft_inference = sft["messages"][0]["content"] + "\n\n" + sft["messages"][1]["content"]

variants = {
    "A/B — Zero-shot": zs,
    "C — Few-shot (3 examples)": fs3,
    "D — SFT inference format": sft_inference,
}

print(f"Token comparison for record ID={sample_rec['id']} ({sample_rec['category']}):\n")
for name, prompt in variants.items():
    print(f"  {name:35s} {count_tokens(prompt):>5} tokens  ({len(prompt):>5} chars)")

Token comparison for record ID=12899063 (Computers And Accessories):

  A/B — Zero-shot                       384 tokens  ( 1417 chars)
  C — Few-shot (3 examples)             951 tokens  ( 3268 chars)
  D — SFT inference format              350 tokens  ( 1303 chars)


## 8. Save prompt artifacts

Save the templates and JSON schemas so downstream notebooks and `ave/prompts/templates.py`
can load them directly.

In [20]:
# Save JSON schemas for constrained decoding (one per category)
json_schemas = {}
for cat in CATEGORY_SCHEMAS:
    json_schemas[cat] = build_json_schema(cat, CATEGORY_SCHEMAS)

schemas_path = CLEANED_DIR / "json_schemas_for_decoding.json"
with open(schemas_path, "w") as f:
    json.dump(json_schemas, f, indent=2)
print(f"Saved JSON schemas to {schemas_path}")

# Save prompt templates as text files for reference
templates_dir = CLEANED_DIR / "prompt_templates"
templates_dir.mkdir(exist_ok=True)

with open(templates_dir / "zero_shot.txt", "w") as f:
    f.write(ZERO_SHOT_TEMPLATE_STR)
with open(templates_dir / "few_shot.txt", "w") as f:
    f.write(FEW_SHOT_TEMPLATE_STR)
with open(templates_dir / "sft_system_message.txt", "w") as f:
    f.write(SYSTEM_MESSAGE)
with open(templates_dir / "sft_user_template.txt", "w") as f:
    f.write(USER_TEMPLATE_STR)

print(f"Saved prompt templates to {templates_dir}/")
for p in sorted(templates_dir.iterdir()):
    print(f"  {p.name}")

Saved JSON schemas to ../data/WDC-PAVE/cleaned/json_schemas_for_decoding.json
Saved prompt templates to ../data/WDC-PAVE/cleaned/prompt_templates/
  few_shot.txt
  sft_system_message.txt
  sft_user_template.txt
  zero_shot.txt


## Summary

**Templates built and tested:**

| Variant | Prompt type | Key difference |
|---------|-------------|----------------|
| A | Zero-shot | Bare task + rules + input |
| B | Zero-shot + constrained decoding | Same prompt, JSON Schema forces valid output |
| C | Few-shot (2–5 examples) | Adds same-category examples from train split |
| D | SFT chat format | system/user/assistant turns for fine-tuning |
| E | Same as A (or C) | Sent to proprietary API |

**Token budget findings:**
- Zero-shot prompts: check the stats above for per-category means
- Few-shot (3 examples): adds significant overhead — check the overhead table
- Few-shot (5 examples): verify context window fit from the headroom check
- SFT examples: check max total tokens for `max_seq_len` setting

**Decisions made:**
- Few-shot selection: **same-category, diversity-sampled** (evenly spaced by fill ratio)
- SFT format: **chat-style** (system + user + assistant) — compatible with TRL/Axolotl
- JSON Schema: **built and validated** against all gold JSONs — ready for constrained decoding
- No lowercasing in prompts — matches the canonicalization decision from notebook 02

**Output files:**
- `data/WDC-PAVE/cleaned/json_schemas_for_decoding.json` — per-category JSON Schemas
- `data/WDC-PAVE/cleaned/prompt_templates/` — all template text files for reference